# GC-LSTM-GhostNet - Step 4 sampled training smoke test

Train GCN → LSTM → temporal attention → GhostNet with leakage-safe contiguous windows.


In [ ]:
from pathlib import Path
import base64
import io
import json
import os
import shutil
import subprocess
import sys
import zipfile
from kaggle_secrets import UserSecretsClient

PROJECT_DIR = Path("/kaggle/working/Luan-Van-GC-LSTM-GhostNet-CICDDoS2019-v1")
OUTPUT_DIR = PROJECT_DIR / "outputs" / "step4_smoke"
MOUNTED_DATA_CANDIDATES = [
    Path("/kaggle/input/cicddos2019-parquet"),
    Path("/kaggle/input/datasets/dungnguyen28101991/cicddos2019-parquet"),
]
DOWNLOADED_DATA_DIR = Path("/kaggle/working/cicddos2019-parquet-input")
PROJECT_ARCHIVE_B64 = "..."

if PROJECT_DIR.exists():
    shutil.rmtree(PROJECT_DIR)
PROJECT_DIR.mkdir(parents=True)
with zipfile.ZipFile(io.BytesIO(base64.b64decode(PROJECT_ARCHIVE_B64))) as project_zip:
    project_zip.extractall(PROJECT_DIR)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(PROJECT_DIR / "requirements.txt")],
    check=True,
)

mounted_data_dir = next((path for path in MOUNTED_DATA_CANDIDATES if path.exists()), None)
if mounted_data_dir is None and next(Path("/kaggle/input").rglob("dataset_summary.json"), None):
    # Kaggle may choose a normalized mount slug that differs from the API slug.
    # The data loader recursively selects the validated manifests below this root.
    mounted_data_dir = Path("/kaggle/input")
if mounted_data_dir is not None:
    DATA_DIR = mounted_data_dir
else:
    DATA_DIR = DOWNLOADED_DATA_DIR
    if DATA_DIR.exists():
        shutil.rmtree(DATA_DIR)
    DATA_DIR.mkdir(parents=True)
    download_env = os.environ.copy()
    secret_value = UserSecretsClient().get_secret("KAGGLE_API_TOKEN")
    try:
        classic = json.loads(secret_value)
    except (TypeError, json.JSONDecodeError):
        download_env["KAGGLE_API_TOKEN"] = secret_value
    else:
        download_env["KAGGLE_USERNAME"] = classic["username"]
        download_env["KAGGLE_KEY"] = classic["key"]
    subprocess.run(
        ["kaggle", "datasets", "download", "-d", "dungnguyen28101991/cicddos2019-parquet",
         "-p", str(DATA_DIR), "--unzip", "--quiet"],
        env=download_env,
        check=True,
    )
    for key in ("KAGGLE_API_TOKEN", "KAGGLE_USERNAME", "KAGGLE_KEY"):
        download_env.pop(key, None)
    del secret_value
print(f"Step 4 project ready; using dataset at {DATA_DIR}")


In [ ]:
command = [
    sys.executable, "train.py",
    "--data-dir", str(DATA_DIR),
    "--output-dir", str(OUTPUT_DIR),
    "--config", "configs/base.yaml",
    "--mode-config", "configs/practical_baseline.yaml",
    "--samples-per-file", "2048",
    "--sequence-length", "16",
    "--sequence-stride", "8",
    "--epochs", "2",
    "--batch-size", "64",
    "--device", "cpu",
    "--run-name", "kaggle-step4-smoke",
]
subprocess.run(command, cwd=PROJECT_DIR, check=True)


In [ ]:
summary_path = OUTPUT_DIR / "step4_summary.json"
summary = json.loads(summary_path.read_text(encoding="utf-8"))
assert summary["status"] == "passed", summary
assert summary["sequence_leakage_status"] == "passed", summary
assert summary["best_epoch"] in (1, 2), summary
assert (OUTPUT_DIR / "training" / "best_model.pt").exists()
assert (OUTPUT_DIR / "training" / "test_metrics.json").exists()
summary
